# 3.8 · 图像特征基础 / Image Feature Basics

> **课程定位 / Where this fits**
> 第 8 课，**Part 3 · EDA 与数据预处理**。
> Lesson 8, **Part 3 · EDA & Preprocessing**.
>
> 文本之后是图像。这一课讲**深度学习之前**人们怎么从图像里提特征：把像素展平、HOG（梯度方向直方图）、颜色直方图等。理解这些"手工特征"的局限，才能真正理解**为什么 CNN 要自动学特征**——CNN 第一层学到的，其实就是边缘检测器，和 HOG 同源。
> After text comes images. This lesson covers how features were extracted from images **before deep learning**: flattening pixels, HOG (histogram of oriented gradients), color histograms. Understanding these hand-crafted features' limits is the key to understanding **why CNNs learn features automatically** — a CNN's first layer learns edge detectors, the same idea as HOG.
>
> 💼 **实战/面试视角**："为什么不直接用像素 / 传统图像特征 / CNN 在自动学什么" 偏 CV/深度学习岗。
> 💼 **Practical/interview angle:** "why not raw pixels / classic image features / what CNNs learn" — CV/DL roles.

> 💡 **面试相关 / Interview-relevant**
> - "为什么不能直接把像素喂模型（维度/无不变性）"（出镜率 ★★★★）
> - "HOG 是什么 / 为什么对光照、平移更鲁棒"（★★★★）
> - "传统特征 vs CNN：CNN 在自动化什么"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 理解**像素展平**的做法及其致命缺陷（维度爆炸 + 无不变性）。
   Understand pixel flattening and its fatal flaws (dimension blow-up + no invariance).
2. 用 **HOG** 提取边缘方向特征，理解它为何更鲁棒。
   Extract HOG edge-orientation features and see why they're more robust.
3. 了解颜色/亮度直方图及"特征要匹配任务"。
   Know color/intensity histograms and "features must match the task".
4. 看清 **Sobel 边缘 = HOG 梯度 = CNN 第一层**的同源关系。
   See that Sobel edges = HOG gradients = a CNN's first layer.

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [像素展平及其缺陷 ⭐](#2)
3. [HOG：梯度方向直方图 ⭐](#3)
4. [HOG vs 像素：分类对比 ⭐](#4)
5. [亮度直方图：特征要匹配任务](#5)
6. [边缘检测 = CNN 在做的事 ⭐](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

最朴素的想法：一张图就是一堆像素，**把它们排成一列**当特征向量不就行了？对 8×8 的小图勉强能用，但对真实图像（224×224×3 ≈ 15 万维）会**维度爆炸**，而且像素特征有个致命问题：**没有不变性**——同一个物体平移一点点、亮度变一点点，像素向量就完全不同，模型会以为是另一张图。
The naive idea: an image is just pixels, so **stack them into one vector** as features. For an 8×8 image it barely works, but for real images (224×224×3 ≈ 150k dims) it **blows up**, and pixel features have a fatal flaw: **no invariance** — shift the object slightly or change brightness, and the pixel vector is totally different, so the model thinks it's a new image.

我们用 **Digits**（8×8 手写数字）演示，因为它够小、能直接看清。
We use **Digits** (8×8 handwritten digits) — small enough to see clearly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
rng = np.random.default_rng(42)

digits = load_digits()
print(f"图像数 images: {len(digits.images)}")
print(f"单张图 one image: {digits.images[0].shape} (8×8 灰度, 值 0-16)")
print(f"展平后 flattened: {digits.data.shape[1]} 维像素向量")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab in zip(axes, digits.images[:8], digits.target[:8]):
    ax.imshow(img, cmap="gray_r"); ax.set_title(f"label: {lab}"); ax.axis("off")
plt.suptitle("手写数字 handwritten digits (8×8)", y=1.1); plt.tight_layout(); plt.show()


<a id="2"></a>
## 2. 像素展平及其缺陷 ⭐ / Pixel Flattening & Its Flaws

把 8×8 图展平成 64 维向量，直接喂分类器。在这么小的、对齐良好的图上还能跑出不错的准确率。但下面演示**无平移不变性**：把同一张图右移 1 像素，像素向量就和原图差出可观的距离——模型对这点微小变化极不鲁棒。
Flatten the 8×8 image into a 64-D vector and feed a classifier. On such small, well-aligned images it works decently. But below shows the **lack of translation invariance**: shift the same image by 1 pixel and its pixel vector differs substantially from the original — the model is fragile to this tiny change.


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_raw, y = digits.data, digits.target   # (1797, 64) 像素向量 / pixel vectors
acc_raw = cross_val_score(make_pipeline(StandardScaler(), SVC()), X_raw, y, cv=5).mean()
print(f"原始像素(64维) + SVM: CV 准确率 = {acc_raw:.1%}")
print("8×8 小图勉强能用; 放到 224×224 真实图就崩(15万维 + 无不变性)")

# 演示无平移不变性: 把图右移 1 像素, 看像素向量距离变化 / shift breaks pixel features
img = digits.images[0]
shifted = np.roll(img, 1, axis=1)        # np.roll 把图整体右移 1 列(像素)
dist_shift = np.linalg.norm(img.ravel() - shifted.ravel())          # 自己 vs 平移版的距离
dist_other = np.linalg.norm(img.ravel() - digits.images[1].ravel()) # 自己 vs 另一个数字的距离
print(f"\n同一张图右移1像素后, 像素向量距离 = {dist_shift:.1f}")
print(f"这张图 vs 另一个不同数字的距离 = {dist_other:.1f}")
print("右移这么小的变化就产生可观距离 → 像素特征对平移极不鲁棒")


<a id="3"></a>
## 3. HOG：梯度方向直方图 ⭐ / HOG: Histogram of Oriented Gradients

**HOG** 的核心思想：不看"每个像素多亮"，而看"**边缘往哪个方向走**"。它把图分成小格子，在每个格子里统计梯度（亮度变化）的方向分布。因为数字/物体的本质是**形状（边缘）**而非绝对亮度，HOG 对**光照变化**和**小幅平移**更鲁棒，曾是行人检测等任务的金标准。
The core idea of **HOG**: instead of "how bright is each pixel", look at "**which way do the edges point**". It divides the image into cells and, in each cell, histograms the gradient (brightness-change) directions. Since a digit/object is fundamentally about **shape (edges)** not absolute brightness, HOG is more robust to **lighting changes** and **small shifts** — long the gold standard for pedestrian detection.


In [ ]:
from skimage.feature import hog
from skimage import exposure

img = digits.images[0]
# hog 返回特征向量; visualize=True 还返回可视化图 / HOG features + a visualization image
features, hog_image = hog(img, orientations=8, pixels_per_cell=(2,2),
                          cells_per_block=(1,1), visualize=True)
print(f"8×8 图 → HOG 特征 {len(features)} 维 (统计了各格子里边缘的方向分布)")

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap="gray_r"); axes[0].set_title("原始数字 original"); axes[0].axis("off")
hog_vis = exposure.rescale_intensity(hog_image, in_range=(0, hog_image.max()))  # 增强对比便于看
axes[1].imshow(hog_vis, cmap="gray"); axes[1].set_title("HOG 可视化(边缘方向)"); axes[1].axis("off")
plt.tight_layout(); plt.show()
print("HOG 图显示了笔画的边缘方向 — 这才是'数字形状'的本质, 而非逐像素亮度")


<a id="4"></a>
## 4. HOG vs 像素：分类对比 ⭐ / HOG vs Pixels

把每张图都转成 HOG 特征，再训同样的分类器，和像素法对比。在这个**小而对齐良好**的数据集上两者接近；但在真实场景（光照、平移、尺度变化大）里，HOG 的鲁棒性优势会明显得多。
Convert every image to HOG features and train the same classifier, comparing to pixels. On this **small, well-aligned** dataset they're close; but in real scenes (big lighting/translation/scale variation), HOG's robustness advantage is much larger.


In [ ]:
# 对所有图算 HOG 特征 / compute HOG for all images
X_hog = np.array([hog(img, orientations=8, pixels_per_cell=(2,2), cells_per_block=(1,1))
                  for img in digits.images])
print(f"全部图的 HOG 特征矩阵: {X_hog.shape}")

acc_hog = cross_val_score(make_pipeline(StandardScaler(), SVC()), X_hog, y, cv=5).mean()
print(f"\n原始像素 + SVM: {acc_raw:.1%}")
print(f"HOG 特征 + SVM: {acc_hog:.1%}")
print("HOG 用边缘方向, 在这个小数据上和像素接近; 真实场景(光照/平移大)优势明显")


<a id="5"></a>
## 5. 亮度直方图：特征要匹配任务 / Histograms: Match Feature to Task

**颜色/亮度直方图**统计"各亮度值出现了多少次"，对**颜色/纹理**任务有用，但**丢掉了空间信息**。下面看一个反例：数字 0 和 8 的亮度直方图很像（都有很多笔画像素），所以直方图**分不开形状相近的数字**。这说明：**特征必须匹配任务**——形状任务用 HOG，颜色任务用颜色直方图。
**Color/intensity histograms** count "how often each brightness value appears", useful for **color/texture** tasks but **discarding spatial info**. A counterexample below: digits 0 and 8 have similar intensity histograms (both have many stroke pixels), so histograms **can't separate similarly-shaped digits**. Lesson: **features must match the task** — HOG for shape, color histograms for color.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5))
for col, dig in enumerate([0, 1, 8, 7]):
    idx = np.where(digits.target == dig)[0][0]
    axes[0, col].imshow(digits.images[idx], cmap="gray_r")
    axes[0, col].set_title(f"digit {dig}"); axes[0, col].axis("off")
    # 亮度直方图: 把像素值(0-16)分到 17 个 bin 计数 / intensity histogram
    hist, _ = np.histogram(digits.images[idx].ravel(), bins=17, range=(0, 16))
    axes[1, col].bar(range(17), hist); axes[1, col].set_title("亮度直方图 intensity hist")
plt.tight_layout(); plt.show()
print("0 和 8 的亮度直方图相似(都有很多笔画) → 颜色/亮度直方图分不开形状相近的数字")
print("→ 形状任务用 HOG, 颜色任务用颜色直方图 — 特征要匹配任务")


<a id="6"></a>
## 6. 边缘检测 = CNN 在做的事 ⭐ / Edge Detection = What CNNs Do

最后一个串联：HOG 的第一步是算**梯度（边缘）**。用 **Sobel 算子**就能提取水平/垂直边缘。而**CNN 第一层卷积核**经过训练后，学到的恰恰就是这类**边缘检测器**——只不过它是**自动学**出来的，不用人手设计。所以理解传统特征，就理解了深度学习在**自动化**什么。
A final connection: HOG's first step computes **gradients (edges)**. A **Sobel operator** extracts horizontal/vertical edges. And a trained **CNN's first-layer filters** learn exactly these kinds of **edge detectors** — except **learned automatically**, no hand-design. So understanding hand-crafted features tells you what deep learning is **automating**.


In [ ]:
from scipy import ndimage
img = digits.images[0].astype(float)
sobel_x = ndimage.sobel(img, axis=1)    # 水平方向梯度 → 检测竖直边缘 / horizontal gradient
sobel_y = ndimage.sobel(img, axis=0)    # 垂直方向梯度 → 检测水平边缘 / vertical gradient

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img, cmap="gray_r"); axes[0].set_title("原图 original"); axes[0].axis("off")
axes[1].imshow(np.abs(sobel_x), cmap="gray"); axes[1].set_title("水平边缘 Sobel-x"); axes[1].axis("off")
axes[2].imshow(np.abs(sobel_y), cmap="gray"); axes[2].set_title("垂直边缘 Sobel-y"); axes[2].axis("off")
plt.tight_layout(); plt.show()
print("Sobel 边缘检测 = HOG 的梯度步骤 = CNN 第一层卷积核学到的东西")
print("理解传统手工特征 → 理解深度学习(CNN)在自动化什么")


<a id="7"></a>
## 7. 小结 / Summary

```
像素展平: 最朴素; 缺陷=维度爆炸(224²×3≈15万) + 无不变性(平移/光照一变就崩)
HOG: 统计各格子的梯度方向 → 抓"形状/边缘"而非亮度 → 对光照/小平移鲁棒
亮度/颜色直方图: 抓颜色纹理但丢空间信息; 0 和 8 直方图相似 → 特征要匹配任务
Sobel 边缘 = HOG 梯度步骤 = CNN 第一层卷积核(自动学的边缘检测器)
传统特征是手工设计; CNN(后续部分)自动学这些特征 → 这是深度学习的核心进步
```

### 💡 面试速查 / Interview cheat-sheet
1. **别直接用像素**：维度爆炸 + 无平移/光照不变性。
   Don't use raw pixels: dimension blow-up + no translation/lighting invariance.
2. **HOG 抓边缘方向**（形状），对光照/小平移更鲁棒。
   HOG captures edge orientations (shape), robust to lighting/small shifts.
3. **特征要匹配任务**：形状→HOG，颜色→颜色直方图。
   Match feature to task: shape→HOG, color→color histogram.
4. **CNN 第一层 = 自动学到的边缘检测器**（≈ Sobel/HOG）。
   A CNN's first layer = automatically learned edge detectors (≈ Sobel/HOG).
5. 传统特征**手工设计**，CNN **自动学习**——这是深度学习的核心价值。
   Hand-crafted vs learned features — the core value of deep learning.

### 下一节 / Next
**3.9 数据泄漏**——前几课反复提到的"防泄漏"，这一课系统讲：什么是泄漏、有哪些常见形式、怎么用正确的流程根除它。
**3.9 Data Leakage** — the recurring "prevent leakage" theme, now systematic: what leakage is, its common forms, and how the right workflow eliminates it.
